# 120 years of Olympic history: athlete participation and medals

This notebook explores the [Kaggle *120 years of Olympic history: athletes and results* dataset](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results). Its source covers observed modern Games from Athens 1896 through Rio 2016. Run the cells from top to bottom after placing `athlete_events.csv` and `noc_regions.csv` in `data/`; nothing is downloaded automatically.

## Questions and units of analysis

1. Which NOCs supplied the most **distinct athletes** at the first and last observed Games, overall and within each season? How did those NOCs' annual participation change?
2. Which athletes have the most gold and total medal awards? Which sports have the fewest distinct medal-winning athletes?
3. How many athletes represented a different NOC at their next Olympic appearance, by year?
4. How did participation by sex and the size of the sports/events programme change?

`athlete_events.csv` has **one row per athlete-event**, not one row per person or team. `ID` identifies a person; `Games` identifies an edition (year and season). Participation counts use unique `ID` values within the stated Games, year, NOC, or subgroup. Medal counts use distinct athlete–Games–Event awards. NOC codes are the primary team identifiers; the region lookup supplies readable labels only and may be historically imperfect.

## 1. Load and inspect the source

Run this notebook from the repository root. The next cell names every missing required file, so setup failures are actionable.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)

data_dir = Path.cwd() / "data"
athlete_path = data_dir / "athlete_events.csv"
region_path = data_dir / "noc_regions.csv"
missing_files = [str(path) for path in (athlete_path, region_path) if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Missing required dataset file(s): " + ", ".join(missing_files)
        + ". Download both CSVs from the Kaggle link above and place them in data/."
    )

raw = pd.read_csv(athlete_path, low_memory=False)
regions_raw = pd.read_csv(region_path, low_memory=False)
required = {"ID", "Name", "NOC", "Games", "Year", "Season", "Sport", "Event", "Medal"}
missing_columns = sorted(required - set(raw.columns))
if missing_columns:
    raise ValueError(f"athlete_events.csv is missing required columns: {missing_columns}")
if not {"NOC", "region"}.issubset(regions_raw.columns):
    raise ValueError("noc_regions.csv must contain NOC and region columns")

print(f"athlete_events.csv: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"noc_regions.csv: {regions_raw.shape[0]:,} rows × {regions_raw.shape[1]} columns")
display(raw.head())

In [ ]:
print("Columns and inferred types:")
display(raw.dtypes.rename("dtype").to_frame())
print(f"Exact duplicate athlete-event rows: {raw.duplicated().sum():,}")
print("Missing values by column (raw file):")
display(raw.isna().sum().rename("missing rows").to_frame())

observed_games_raw = (
    raw[["Games", "Year", "Season"]]
    .drop_duplicates()
    .assign(season_order=lambda d: d["Season"].map({"Winter": 0, "Summer": 1}))
    .sort_values(["Year", "season_order", "Games"])
    .drop(columns="season_order")
    .reset_index(drop=True)
)
print(f"Observed Games: {len(observed_games_raw)}")
display(observed_games_raw)

## 2. Prepare analysis fields

Only exact duplicate rows are removed globally. Blank demographic fields (`Age`, `Height`, `Weight`, or `Sex`) and missing `Medal` values do **not** invalidate an athlete-event row. Missing medals mean no recorded award. Each calculation filters only the keys it needs: for example, participation needs an `ID` and `NOC`, while medal analysis also needs `Event` and an award color. Empty text is treated as missing; no demographic values are imputed. A missing NOC is not assigned a country.

The region mapping is descriptive, not a replacement for NOC. Historically distinct NOCs can map to the same modern region, and some teams may lack a region label. Where possible, charts retain the code alongside the readable label.

In [ ]:
events = raw.drop_duplicates().copy()
for column in ["Name", "NOC", "Games", "Season", "Sport", "Event", "Medal", "Sex"]:
    if column in events:
        events[column] = events[column].astype("string").str.strip().replace("", pd.NA)

regions = regions_raw.copy()
for column in ["NOC", "region"]:
    regions[column] = regions[column].astype("string").str.strip().replace("", pd.NA)
duplicate_region_codes = regions["NOC"].dropna().duplicated().sum()
region_lookup = (
    regions.dropna(subset=["NOC", "region"])
    .drop_duplicates(subset="NOC", keep="first")
    .set_index("NOC")["region"]
    .to_dict()
)

def noc_label(noc):
    region = region_lookup.get(noc)
    return f"{noc} ({region})" if pd.notna(region) else str(noc)

games = (
    events.dropna(subset=["Games", "Year", "Season"])[["Games", "Year", "Season"]]
    .drop_duplicates()
    .assign(season_order=lambda d: d["Season"].map({"Winter": 0, "Summer": 1}))
    .sort_values(["Year", "season_order", "Games"])
    .drop(columns="season_order")
    .reset_index(drop=True)
)
if games.empty:
    raise ValueError("No Games have nonmissing Games, Year, and Season values")

print(f"Removed {len(raw) - len(events):,} exact duplicate row(s); retained {len(events):,} rows.")
print(f"Duplicate NOC keys in region file (first label retained): {duplicate_region_codes:,}")
unmapped_codes = sorted(set(events["NOC"].dropna()) - set(region_lookup))
print(f"NOC codes without a region label: {len(unmapped_codes)}")
print("Examples:", unmapped_codes[:15])
display(events[["ID", "Games", "Year", "Season", "NOC", "Sport", "Event", "Medal"]].isna().sum().rename("missing after preparation").to_frame())

## 3. First and last observed Games: NOC participation

The endpoints come from the observed `Games` values, sorted by year and then Winter before Summer in a shared year. The top five are ranked by `ID.nunique()` **within that Games**. Equal counts at the fifth position are ordered alphabetically by NOC so each list contains five codes. Annual trends count each ID once per NOC and year across all observed Games in that year; a zero means that NOC has no recorded athlete in an observed year.

In [ ]:
def endpoint_rankings(frame):
    editions = (
        frame.dropna(subset=["Games", "Year", "Season"])[["Games", "Year", "Season"]]
        .drop_duplicates()
        .assign(season_order=lambda d: d["Season"].map({"Winter": 0, "Summer": 1}))
        .sort_values(["Year", "season_order", "Games"])
    )
    if editions.empty:
        raise ValueError("No observed Games in this subset")
    first_game, last_game = editions.iloc[0]["Games"], editions.iloc[-1]["Games"]

    def rank(game):
        ranked = (
            frame.loc[frame["Games"].eq(game)]
            .dropna(subset=["ID", "NOC"])
            .groupby("NOC", as_index=False)["ID"]
            .nunique()
            .rename(columns={"ID": "Unique athletes"})
            .sort_values(["Unique athletes", "NOC"], ascending=[False, True])
            .head(5)
            .reset_index(drop=True)
        )
        ranked.insert(1, "Region label", ranked["NOC"].map(region_lookup))
        return ranked

    return first_game, last_game, rank(first_game), rank(last_game)


def readable_year_ticks(ax, years):
    years = sorted(set(int(year) for year in years))
    if years:
        stride = max(1, (len(years) + 10) // 11)
        ticks = years[::stride]
        if years[-1] not in ticks:
            ticks.append(years[-1])
        ax.set_xticks(ticks)
        ax.tick_params(axis="x", rotation=45)


def show_endpoint_analysis(frame, scope):
    first_game, last_game, first_rank, last_rank = endpoint_rankings(frame)
    display(Markdown(f"### {scope}: {first_game} and {last_game}"))
    display(Markdown(f"**{first_game} — five most represented NOCs**"))
    display(first_rank)
    display(Markdown(f"**{last_game} — five most represented NOCs**"))
    display(last_rank)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
    for ax, game, ranking in zip(axes, [first_game, last_game], [first_rank, last_rank]):
        bars = ax.barh(ranking["NOC"].iloc[::-1], ranking["Unique athletes"].iloc[::-1], color="#38678f")
        ax.bar_label(bars, padding=3, fmt="%.0f")
        ax.set_xlim(0, max(ranking["Unique athletes"].max() * 1.18, 1))
        ax.set_title(f"{scope}: top five NOCs in {game}")
        ax.set_xlabel("Distinct athletes (ID)")
        ax.set_ylabel("NOC")
    plt.show()

    all_codes = list(dict.fromkeys(first_rank["NOC"].tolist() + last_rank["NOC"].tolist()))
    years = sorted(frame["Year"].dropna().astype(int).unique())
    annual = (
        frame.dropna(subset=["ID", "NOC", "Year"])
        .groupby(["Year", "NOC"])["ID"].nunique()
    )
    palette = dict(zip(all_codes, sns.color_palette("tab10", n_colors=len(all_codes))))
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True, layout="constrained")
    for ax, group_name, ranking in zip(axes, ["First-Games group", "Last-Games group"], [first_rank, last_rank]):
        for noc in ranking["NOC"]:
            values = annual.reindex(pd.MultiIndex.from_product([years, [noc]]), fill_value=0).to_numpy()
            ax.plot(years, values, marker="o", markersize=3, linewidth=1.8, label=noc_label(noc), color=palette[noc])
        ax.set_title(f"{scope}: {group_name} ({first_game if group_name.startswith('First') else last_game})")
        ax.set_xlabel("Observed Olympic year")
        ax.set_ylabel("Distinct athletes (ID) per NOC and year")
        readable_year_ticks(ax, years)
        ax.legend(title="NOC (region label)", fontsize=8, loc="best")
    plt.show()

    overlap = set(first_rank["NOC"]) & set(last_rank["NOC"])
    display(Markdown(
        f"**Reading these charts.** {first_game}'s leader is {noc_label(first_rank.iloc[0]['NOC'])} "
        f"with {first_rank.iloc[0]['Unique athletes']:,} distinct athletes; {last_game}'s leader is "
        f"{noc_label(last_rank.iloc[0]['NOC'])} with {last_rank.iloc[0]['Unique athletes']:,}. "
        f"The two top-five lists share {len(overlap)} NOC code(s). Lines show recorded participation, "
        "not population-adjusted rates; NOC histories and Games programmes can differ."
    ))
    return {"first_game": first_game, "last_game": last_game, "first": first_rank, "last": last_rank, "overlap": overlap}


overall = show_endpoint_analysis(events, "All Games")

## 4. Summer and Winter comparisons

Each season gets its **own** first and last observed Games. Seasonal lines use only that season's observed years, so shared-year Games before 1994 and staggered Games afterward are handled from the source rather than from a presumed schedule. Compare counts within each season; the different sizes and sports programmes make direct size comparisons between Summer and Winter descriptive only.

In [ ]:
season_results = {}
for season in ["Summer", "Winter"]:
    season_results[season] = show_endpoint_analysis(events.loc[events["Season"].eq(season)], season)

comparison = pd.DataFrame([
    {
        "Season": season,
        "First observed Games": result["first_game"],
        "First leader": result["first"].iloc[0]["NOC"],
        "Last observed Games": result["last_game"],
        "Last leader": result["last"].iloc[0]["NOC"],
        "Top-five overlap": len(result["overlap"]),
    }
    for season, result in season_results.items()
])
display(Markdown("**Season comparison of endpoint rankings**"))
display(comparison)

## 5. Athlete medal records

An award is one distinct `(ID, Games, Event)` with a recorded Gold, Silver, or Bronze medal. If the source repeats an athlete's event row, it is counted once. Each team member's recorded award counts for that athlete; these are **person-awards**, not team medals or NOC medal-table totals. We check that the same athlete–Games–Event does not carry conflicting medal colors. Athlete IDs are authoritative when names vary.

In [ ]:
award_rows = events.loc[
    events["Medal"].isin(["Gold", "Silver", "Bronze"])
    & events[["ID", "Games", "Event"]].notna().all(axis=1),
    ["ID", "Name", "Games", "Event", "Sport", "Medal"],
].copy()
conflicting_awards = award_rows.groupby(["ID", "Games", "Event"])["Medal"].nunique().gt(1).sum()
if conflicting_awards:
    raise ValueError(f"Found {conflicting_awards} athlete–Games–Event award(s) with conflicting medal colors")
awards = award_rows.drop_duplicates(subset=["ID", "Games", "Event"])
name_lookup = events.dropna(subset=["ID", "Name"]).drop_duplicates("ID").set_index("ID")["Name"]

def medal_leaders(counts, count_label):
    maximum = counts.max()
    leaders = counts[counts.eq(maximum)].rename(count_label).reset_index()
    leaders.insert(1, "Name", leaders["ID"].map(name_lookup))
    return leaders.sort_values(["Name", "ID"], na_position="last").reset_index(drop=True)

gold_counts = awards.loc[awards["Medal"].eq("Gold")].groupby("ID").size()
all_medal_counts = awards.groupby("ID").size()
if gold_counts.empty or all_medal_counts.empty:
    raise ValueError("No recorded medal awards are available for the athlete leaderboards")

gold_leaders = medal_leaders(gold_counts, "Gold awards")
total_leaders = medal_leaders(all_medal_counts, "Total awards")
display(Markdown("**Most gold awards — all ties**"))
display(gold_leaders)
display(Markdown("**Most total awards — all ties**"))
display(total_leaders)
print(f"Distinct athlete–Games–Event awards analyzed: {len(awards):,}")

## 6. Athletes changing NOC

First, collapse event rows to one `(ID, Games, NOC)` record. Then make one ordered Games appearance per athlete, compare its NOC with the **immediately previous** appearance, and count distinct switching IDs by year. First-time Olympians have no previous appearance and are excluded. When Summer and Winter share a year, **Winter precedes Summer**. If an athlete has multiple NOCs within the same Games, that appearance cannot have one unambiguous code; comparisons involving it are excluded, while the appearance stays in sequence. Missing NOC values are handled the same way. An athlete switching twice in one calendar year contributes only one to that year's count.

In [ ]:
appearance_noc = (
    events.dropna(subset=["ID", "Games", "Year", "Season"])
    [["ID", "Games", "Year", "Season", "NOC"]]
    .drop_duplicates(subset=["ID", "Games", "NOC"])
)
appearances = (
    appearance_noc.groupby(["ID", "Games", "Year", "Season"], as_index=False)
    .agg(NOC=("NOC", "first"), noc_count=("NOC", "nunique"),
         missing_noc=("NOC", lambda values: values.isna().any()))
)
appearances["season_order"] = appearances["Season"].map({"Winter": 0, "Summer": 1})
appearances = appearances.sort_values(["ID", "Year", "season_order", "Games"]).reset_index(drop=True)
appearances["previous_NOC"] = appearances.groupby("ID")["NOC"].shift()
appearances["unambiguous"] = appearances["noc_count"].eq(1) & ~appearances["missing_noc"]
appearances["previous_unambiguous"] = appearances.groupby("ID")["unambiguous"].shift()
appearances["comparable"] = appearances["unambiguous"] & appearances["previous_unambiguous"].fillna(False)
appearances["switched"] = appearances["comparable"] & appearances["NOC"].ne(appearances["previous_NOC"])
appearances["switched"] = appearances["switched"].fillna(False)

years = sorted(games["Year"].astype(int).unique())
switches_by_year = (
    appearances.loc[appearances["switched"]]
    .groupby("Year")["ID"].nunique()
    .reindex(years, fill_value=0)
    .rename("Distinct athletes changing NOC")
)
display(switches_by_year.to_frame())
fig, ax = plt.subplots(figsize=(12, 4.5), layout="constrained")
ax.plot(switches_by_year.index, switches_by_year.values, marker="o", color="#6f448d")
ax.set(title="Athletes with a different NOC from their previous Olympic appearance", xlabel="Observed Olympic year", ylabel="Distinct athletes (ID)")
readable_year_ticks(ax, years)
plt.show()
print(f"Games appearances with missing or multiple NOC codes: {(~appearances['unambiguous']).sum():,}")
display(Markdown(
    "**Interpretation.** This is a count of recorded code changes between consecutive appearances, "
    "not a measure of nationality changes or their causes. Historical code changes, delegation rules, "
    "and source records can all affect it."
))

## 7. Sports with the fewest distinct medalists

Restrict to sports with at least one recorded award, then count unique medal-winning `ID` values across the full dataset. One athlete can win several awards in a sport but contributes one to that sport's count. Return every sport tied at the minimum.

In [ ]:
medalists_by_sport = (
    awards.dropna(subset=["Sport"])
    .groupby("Sport")["ID"].nunique()
    .sort_values()
    .rename("Distinct medal-winning athletes")
)
if medalists_by_sport.empty:
    raise ValueError("No sports with recorded medal-winning athletes were found")
fewest_medalists = medalists_by_sport[medalists_by_sport.eq(medalists_by_sport.min())].reset_index()
display(Markdown("**All sports tied for the fewest distinct medal-winning athletes**"))
display(fewest_medalists)

smallest_ten = medalists_by_sport.head(10).sort_values()
fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
bars = ax.barh(smallest_ten.index, smallest_ten.values, color="#4b8a83")
ax.bar_label(bars, padding=3)
ax.set_xlim(0, max(smallest_ten.max() * 1.15, 1))
ax.set(title="Sports with the fewest distinct recorded medalists (first ten)", xlabel="Distinct medal-winning athletes (ID)", ylabel="Sport")
plt.show()
display(Markdown("**Interpretation.** A small medalist count may reflect few editions or events, not necessarily a small field in every Games. The table above includes all minimum ties even when a chart displays only ten sports."))

## 8. Supporting historical patterns

These views give context for the endpoint comparisons. Counts remain tied to observed Games; lines connect observations for readability and do not imply values in missing or cancelled years.

### Distinct athletes by Games and season

Count each ID once per Games. This measures recorded participation, regardless of how many events the athlete entered.

In [ ]:
athletes_per_games = (
    events.dropna(subset=["ID", "Games", "Year", "Season"])
    .groupby(["Year", "Season", "Games"], as_index=False)["ID"].nunique()
    .rename(columns={"ID": "Unique athletes"})
)
fig, ax = plt.subplots(figsize=(12, 4.5), layout="constrained")
for season, color in [("Summer", "#d27636"), ("Winter", "#4385a6")]:
    subset = athletes_per_games.loc[athletes_per_games["Season"].eq(season)].sort_values("Year")
    ax.plot(subset["Year"], subset["Unique athletes"], marker="o", markersize=3, label=season, color=color)
ax.set(title="Distinct athletes at each observed Olympic Games", xlabel="Games year", ylabel="Distinct athletes (ID)")
readable_year_ticks(ax, athletes_per_games["Year"])
ax.legend(title="Season")
plt.show()
display(Markdown("**Interpretation.** The two series describe the number of distinct athletes recorded in each season's Games. Differences in programme size and coverage should be considered before comparing levels."))

### Participation by recorded sex

Count each ID once per Games and recorded sex, separately for Summer and Winter. Records with missing or other sex codes are excluded **only** from this breakdown. These are source categories and do not describe athletes' gender identities beyond the recorded field.

In [ ]:
sex_participation = (
    events.loc[events["Sex"].isin(["F", "M"])]
    .dropna(subset=["ID", "Games", "Year", "Season"])
    .groupby(["Year", "Season", "Games", "Sex"], as_index=False)["ID"].nunique()
    .rename(columns={"ID": "Unique athletes"})
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True, layout="constrained")
for ax, season in zip(axes, ["Summer", "Winter"]):
    subset = sex_participation.loc[sex_participation["Season"].eq(season)]
    for sex, color in [("F", "#aa4d86"), ("M", "#3979a9")]:
        series = subset.loc[subset["Sex"].eq(sex)].sort_values("Year")
        ax.plot(series["Year"], series["Unique athletes"], marker="o", markersize=3, label=sex, color=color)
    ax.set(title=f"{season}: distinct athletes by recorded sex", xlabel="Games year", ylabel="Distinct athletes (ID)")
    readable_year_ticks(ax, subset["Year"])
    ax.legend(title="Recorded sex")
plt.show()
display(Markdown("**Interpretation.** Compare the recorded female and male athlete counts within each season over time. Missing sex records are omitted from this view, so these lines need not sum to every Games' total."))

### Size of the recorded sports programme

Count distinct `Sport` and `Event` labels within each Games. These are source labels; naming and programme definitions can change over time.

In [ ]:
programme = (
    events.dropna(subset=["Games", "Year", "Season"])
    .groupby(["Year", "Season", "Games"], as_index=False)
    .agg(Sports=("Sport", "nunique"), Events=("Event", "nunique"))
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
for ax, measure in zip(axes, ["Sports", "Events"]):
    for season, color in [("Summer", "#d27636"), ("Winter", "#4385a6")]:
        subset = programme.loc[programme["Season"].eq(season)].sort_values("Year")
        ax.plot(subset["Year"], subset[measure], marker="o", markersize=3, label=season, color=color)
    ax.set(title=f"Recorded {measure.lower()} at each observed Games", xlabel="Games year", ylabel=f"Distinct {measure.lower()} labels")
    readable_year_ticks(ax, programme["Year"])
    ax.legend(title="Season")
plt.show()
display(Markdown("**Interpretation.** These counts describe the recorded programme at each Games. A change in labels or event definitions can contribute to a change in the count."))

## 9. Findings and limitations

The summary below is calculated at run time from the supplied CSVs. It deliberately avoids hard-coded totals or causal explanations.

In [ ]:
rare_names = ", ".join(fewest_medalists["Sport"].astype(str))
gold_names = ", ".join(f"{row.Name} (ID {row.ID})" for row in gold_leaders.itertuples())
total_names = ", ".join(f"{row.Name} (ID {row.ID})" for row in total_leaders.itertuples())
peak_switch_years = ", ".join(str(year) for year in switches_by_year.index[switches_by_year.eq(switches_by_year.max())])
display(Markdown(
    "### Main findings\n"
    f"- The first and last observed Games are **{overall['first_game']}** and **{overall['last_game']}**. "
    f"Their top-five NOC lists share **{len(overall['overlap'])}** code(s).\n"
    f"- Summer runs from **{season_results['Summer']['first_game']}** to **{season_results['Summer']['last_game']}**; "
    f"Winter runs from **{season_results['Winter']['first_game']}** to **{season_results['Winter']['last_game']}** in this source.\n"
    f"- The maximum recorded gold award count is **{gold_counts.max()}**, held by {gold_names}. "
    f"The maximum total award count is **{all_medal_counts.max()}**, held by {total_names}.\n"
    f"- The fewest distinct medal-winning athletes in a sport is **{medalists_by_sport.min()}**: {rare_names}.\n"
    f"- The highest yearly count of recorded NOC switchers is **{switches_by_year.max()}** in {peak_switch_years}."
))

### Limitations

- **Changing NOCs and imperfect labels.** Historical delegations, border changes, independent athletes, and code reuse complicate country comparisons. `noc_regions.csv` offers readable labels but may map historical teams imperfectly. NOC changes do not by themselves establish a change in citizenship.
- **Missing fields.** Missing demographic data remain in general participation counts; sex-specific charts omit unrecorded sex. Missing IDs, Games identifiers, or NOCs exclude a row only from calculations that require them. Medal blanks are not awards.
- **Athlete-event grain.** One athlete can have several rows in a Games. Raw row counts would overstate participation. Unique `ID` counts are used; event awards are deduplicated by athlete, Games, and Event.
- **Team-event medals.** Every recorded team member's award counts in athlete medal totals. This is not an official count of medals issued to NOCs or teams.
- **Observed schedule and coverage.** Some scheduled Games were cancelled and do not appear. Summer and Winter shared years through 1992 and were staggered afterward. The charts use observed editions only, and this source stops at **2016**; later Olympics are outside scope.
- **Descriptive analysis.** Programme changes, missing records, participation rules, and political history can affect trends. These charts cannot identify a trend's cause.